**Fase 2 — Avaliação de Utilidade Preditiva**

Objetivo: avaliar o impacto das técnicas de anonimização espacial sobre a utilidade preditiva (predição de surtos de Aedes aegypti), com rigor estatístico: 10 seeds × 5 folds × 3 classificadores.

1 - Setup


In [14]:
# ============================================================
# Setup: imports, montagem do Drive e configurações globais
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

import xgboost as xgb

from tqdm.auto import tqdm

# ============================================================
# Paths e constantes globais
# ============================================================

BASE_IN_ORIGINAL = "/content/drive/MyDrive/Mestrado/Dados Gerais"
BASE_ANONIMIZADO = "/content/drive/MyDrive/Mestrado/Dados_Anonimizados"
BASE_OUT_FASE2   = "/content/drive/MyDrive/Mestrado/Resultados_Fase2"

os.makedirs(BASE_OUT_FASE2, exist_ok=True)

LAT_COL = "latitude"
LON_COL = "longitude"

ANOS = [2018, 2019, 2020, 2021, 2022, 2023]

# Técnicas a avaliar (devem corresponder às pastas em BASE_ANONIMIZADO)
TECNICAS = [
    "original",
    "permutacao",
    "generalizacao_dec2",
    "microagregacao_k2", "microagregacao_k5", "microagregacao_k10",
    "dp_eps_0.1", "dp_eps_0.5", "dp_eps_1.0", "dp_eps_2.0", "dp_eps_5.0",
]

TECNICAS_ESTOCASTICAS = {
    "permutacao",
    "microagregacao_k2", "microagregacao_k5", "microagregacao_k10",
    "dp_eps_0.1", "dp_eps_0.5", "dp_eps_1.0", "dp_eps_2.0", "dp_eps_5.0",
}

# Configuração da Fase 2
N_SEEDS_USAR  = 10    # opção B: 10 seeds
N_FOLDS       = 5     # TimeSeriesSplit
N_BOOTSTRAP   = 1000  # para IC 95%
N_REGIOES     = 15    # K-Means clusters

# Classificadores
CLASSIFICADORES = ["xgboost", "random_forest", "logistic_regression"]

print(f"✅ Setup concluído.")
print(f"   Técnicas: {len(TECNICAS)} ({len(TECNICAS_ESTOCASTICAS)} estocásticas)")
print(f"   Seeds: {N_SEEDS_USAR} (estocásticas) × 1 (determinísticas)")
print(f"   Folds: {N_FOLDS}")
print(f"   Classificadores: {len(CLASSIFICADORES)}")
print(f"   Total estimado de fits: "
      f"{len(TECNICAS_ESTOCASTICAS) * N_SEEDS_USAR * N_FOLDS * len(CLASSIFICADORES) + (len(TECNICAS) - len(TECNICAS_ESTOCASTICAS)) * 1 * N_FOLDS * len(CLASSIFICADORES)}")
print(f"   Saída: {BASE_OUT_FASE2}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup concluído.
   Técnicas: 11 (9 estocásticas)
   Seeds: 10 (estocásticas) × 1 (determinísticas)
   Folds: 5
   Classificadores: 3
   Total estimado de fits: 1380
   Saída: /content/drive/MyDrive/Mestrado/Resultados_Fase2


In [15]:
# === Verificar conteúdo atual do parquet de resultados ===
if os.path.exists(CAMINHO_RESULTADOS):
    df_check = pd.read_parquet(CAMINHO_RESULTADOS)
    print(f"📊 Resultados atuais no checkpoint:")
    print(f"   Total de fits: {len(df_check)}")
    print(f"   Técnicas presentes: {sorted(df_check['tecnica'].unique())}")
    print(f"\n   Seeds por técnica:")
    print(df_check.groupby('tecnica')['seed'].nunique().sort_values(ascending=False).to_string())
    print(f"\n   Combinações únicas (técnica × seed × clf): "
          f"{df_check[['tecnica','seed','clf']].drop_duplicates().shape[0]}")

    # Se dec1 estiver lá indevidamente
    if 'generalizacao_dec1' in df_check['tecnica'].unique():
        print(f"\n⚠️  ATENÇÃO: 'generalizacao_dec1' está no checkpoint mas decidimos excluir!")
        print(f"   Vou limpar essas entradas...")
        df_check = df_check[df_check['tecnica'] != 'generalizacao_dec1']
        df_check.to_parquet(CAMINHO_RESULTADOS, index=False)
        print(f"   ✅ Removido. Novo total: {len(df_check)} fits")
else:
    print("Nenhum checkpoint encontrado.")

📊 Resultados atuais no checkpoint:
   Total de fits: 375
   Técnicas presentes: ['dp_eps_0.1', 'dp_eps_0.5', 'dp_eps_1.0', 'dp_eps_2.0', 'dp_eps_5.0', 'generalizacao_dec2', 'microagregacao_k10', 'microagregacao_k2', 'microagregacao_k5', 'original', 'permutacao']

   Seeds por técnica:
tecnica
permutacao            7
dp_eps_0.5            2
dp_eps_0.1            2
dp_eps_1.0            2
dp_eps_2.0            2
microagregacao_k10    2
dp_eps_5.0            2
microagregacao_k5     2
microagregacao_k2     2
generalizacao_dec2    1
original              1

   Combinações únicas (técnica × seed × clf): 75


In [16]:
# ============================================================
# Função: carregar e processar dados climáticos do INMET
# (cálculo único, reaproveitado em todas as técnicas)
# ============================================================

def carregar_clima_features_weekly(base_in=BASE_IN_ORIGINAL, verbose=True):
    """
    Carrega arquivos INMET_*.CSV e produz features climáticas semanais.

    Retorna:
        df_climate_features_weekly : DataFrame indexado por semana,
            contendo as 4 features biológicas:
            - clima_chuva_acum_21d
            - clima_temp_min_media_14d
            - clima_dias_temp_min_20c
            - clima_dias_sem_chuva
    """
    climate_files = sorted(glob.glob(f"{base_in}/INMET_*.CSV"))

    if not climate_files:
        raise FileNotFoundError(
            f"Nenhum arquivo INMET_*.CSV em {base_in}. "
            "Faça o upload dos arquivos antes."
        )

    if verbose:
        print(f"📥 {len(climate_files)} arquivos INMET encontrados")

    # ─── Etapa 1: Concatenar todos os anos ─────────────────────
    all_dfs = []
    for f in climate_files:
        df_c = pd.read_csv(f, sep=';', skiprows=8, decimal=',', encoding='latin-1')
        all_dfs.append(df_c)
    df_climate_raw = pd.concat(all_dfs, ignore_index=True)

    if verbose:
        print(f"   Total bruto: {len(df_climate_raw):,} registros horários")

    # ─── Etapa 2: Padronizar colunas (variam entre anos) ───────
    df_climate = df_climate_raw.copy()
    rename_map = {}
    for col in df_climate.columns:
        col_clean = col.strip()
        if 'DATA (YYYY-MM-DD)' in col_clean:    rename_map[col] = 'data_v1'
        elif col_clean == 'Data':                rename_map[col] = 'data_v2'
        elif 'HORA (UTC)' in col_clean:          rename_map[col] = 'hora_v1'
        elif col_clean == 'Hora UTC':            rename_map[col] = 'hora_v2'
        elif 'PRECIPITAÇÃO' in col_clean:        rename_map[col] = 'precipitacao_mm'
        elif 'TEMPERATURA DO AR' in col_clean:   rename_map[col] = 'temp_c'
        elif 'TEMPERATURA MÍNIMA' in col_clean:  rename_map[col] = 'temp_min_c'
        elif 'TEMPERATURA MÁXIMA' in col_clean:  rename_map[col] = 'temp_max_c'
        elif 'UMIDADE RELATIVA' in col_clean:    rename_map[col] = 'umidade_rel'
    df_climate.rename(columns=rename_map, inplace=True)

    df_climate['data'] = df_climate['data_v1'].combine_first(df_climate['data_v2'])
    df_climate['hora'] = df_climate['hora_v1'].combine_first(df_climate['hora_v2'])

    # ─── Etapa 3: Construir datetime ───────────────────────────
    df_climate['data_str'] = df_climate['data'].astype(str).str.replace('-', '/')
    df_climate['hora_str'] = df_climate['hora'].astype(str).str.replace(' UTC', '').str.zfill(4)
    df_climate['datetime'] = pd.to_datetime(
        df_climate['data_str'] + ' ' + df_climate['hora_str'],
        format='%Y/%m/%d %H%M', errors='coerce'
    )
    df_climate = df_climate.dropna(subset=['datetime'])

    cols = ['datetime', 'precipitacao_mm', 'temp_c', 'temp_min_c']
    df_climate_clean = df_climate[cols].set_index('datetime')
    for c in df_climate_clean.columns:
        df_climate_clean[c] = pd.to_numeric(df_climate_clean[c], errors='coerce')
    df_climate_clean = df_climate_clean.groupby(df_climate_clean.index).mean()

    # ─── Etapa 4: Agregação diária + features biológicas ───────
    df_daily = df_climate_clean.resample('D').agg(
        precip_total_mm=('precipitacao_mm', 'sum'),
        temp_media_c=('temp_c', 'mean'),
        temp_min_c=('temp_min_c', 'min'),
    )
    df_daily = df_daily.interpolate(method='time')

    # Feature 1: chuva acumulada nos 21 dias anteriores
    df_daily['clima_chuva_acum_21d'] = (
        df_daily['precip_total_mm'].rolling(window=21, min_periods=1).sum()
    )

    # Feature 2: temperatura mínima média nos 14 dias anteriores
    df_daily['clima_temp_min_media_14d'] = (
        df_daily['temp_min_c'].rolling(window=14, min_periods=1).mean()
    )

    # Feature 3: dias consecutivos com temp. min. > 20°C
    cond_calor = df_daily['temp_min_c'] > 20
    df_daily['clima_dias_temp_min_20c'] = (
        cond_calor.astype(int)
        .groupby(cond_calor.eq(0).cumsum()).cumsum()
    )

    # Feature 4: dias consecutivos sem chuva
    cond_chuva = df_daily['precip_total_mm'] > 0
    df_daily['clima_dias_sem_chuva'] = (
        cond_chuva.eq(0).astype(int)
        .groupby(cond_chuva.astype(int).cumsum()).cumsum()
    )

    # ─── Etapa 5: Agregação semanal ───────────────────────────
    features_biologicas = [
        'clima_chuva_acum_21d',
        'clima_temp_min_media_14d',
        'clima_dias_temp_min_20c',
        'clima_dias_sem_chuva',
    ]
    df_climate_features_weekly = df_daily[features_biologicas].resample('W').mean()

    if verbose:
        print(f"   Features climáticas semanais: {df_climate_features_weekly.shape}")
        print(f"   Período: {df_climate_features_weekly.index.min().date()} → "
              f"{df_climate_features_weekly.index.max().date()}")

    return df_climate_features_weekly


# Executa uma vez e cacheia em memória
print("🌧️  Carregando dados climáticos do INMET...")
df_climate_features_weekly = carregar_clima_features_weekly(verbose=True)
print(f"\n✅ Features climáticas prontas para o loop principal.")
df_climate_features_weekly.head()

🌧️  Carregando dados climáticos do INMET...
📥 6 arquivos INMET encontrados
   Total bruto: 44,400 registros horários
   Features climáticas semanais: (261, 4)
   Período: 2019-01-06 → 2023-12-31

✅ Features climáticas prontas para o loop principal.


,clima_chuva_acum_21d,clima_temp_min_media_14d,clima_dias_temp_min_20c,clima_dias_sem_chuva
datetime,,,,
2019-01-06,8.833333,22.561944,1.500000,1.000000
2019-01-13,18.085714,22.324900,6.000000,0.571429
2019-01-20,62.400000,22.469388,8.571429,0.000000
2019-01-27,91.285714,21.981633,4.000000,1.714286
2019-02-03,99.400000,22.607143,9.000000,2.142857


2 - Pipeline de Feature Engineering

In [17]:
# ============================================================
# Função: carregar dados de mosquito para uma técnica + seed
# (lê Parquet stacked, filtra pela seed especificada)
# ============================================================

def carregar_mosquito_tecnica_seed(tecnica, seed,
                                    base_anon=BASE_ANONIMIZADO,
                                    anos=None,
                                    verbose=False):
    """
    Carrega dados de mosquito de uma técnica específica, filtrados por seed.

    Parâmetros:
        tecnica : str. Nome da técnica (e.g., "original", "permutacao",
                  "dp_eps_1.0"). Deve corresponder a uma pasta em
                  base_anon/.
        seed    : int. Seed a filtrar (0 para determinísticas; 0..N-1 para
                  estocásticas).
        anos    : list[int] ou None. Anos a carregar. Se None, usa 2019-2023
                  (Opção A: 2018 descartado por ausência de clima INMET).
        base_anon : caminho da pasta de dados anonimizados.

    Retorna:
        df_mosquito : DataFrame com colunas:
            - data_inspecao : datetime
            - latitude, longitude : float
            - total_aedes_aegypti : int
    """
    if anos is None:
        # Opção A: 2019-2023 (2018 descartado por falta de clima INMET)
        anos = [2019, 2020, 2021, 2022, 2023]

    # Lê e empilha por ano, filtrando seed
    dfs_anos = []
    for ano in anos:
        path = f"{base_anon}/{tecnica}/{ano}.parquet"
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"Arquivo não encontrado: {path}. "
                f"A técnica '{tecnica}' não foi gerada para o ano {ano}?"
            )
        df_ano = pd.read_parquet(path)
        df_ano = df_ano[df_ano["seed"] == seed].copy()
        if len(df_ano) == 0:
            raise ValueError(
                f"Nenhum registro encontrado para {tecnica}/{ano} com seed={seed}. "
                f"Seeds disponíveis: {sorted(pd.read_parquet(path)['seed'].unique())}"
            )
        dfs_anos.append(df_ano)

    df = pd.concat(dfs_anos, ignore_index=True)

    # Construir datetime
    df['data_inspecao'] = pd.to_datetime(
        df['inspection_realized_at'],
        format='%d/%m/%Y %H:%M', errors='coerce'
    )

    # Garantir tipos numéricos das coordenadas
    df[LAT_COL] = pd.to_numeric(df[LAT_COL], errors='coerce')
    df[LON_COL] = pd.to_numeric(df[LON_COL], errors='coerce')

    # Contar Aedes aegypti (varre colunas inspection_mosquitoes_*)
    df['total_aedes_aegypti'] = 0
    for i in range(10):
        name_col = f'inspection_mosquitoes_{i}_name'
        qty_col  = f'inspection_mosquitoes_{i}_pivot_quantity'
        if name_col in df.columns and qty_col in df.columns:
            df[qty_col] = pd.to_numeric(df[qty_col], errors='coerce').fillna(0)
            cond_aedes = (df[name_col] == 'Aedes aegypti')
            df.loc[cond_aedes, 'total_aedes_aegypti'] += df.loc[cond_aedes, qty_col]

    # Resultado limpo
    df_limpo = df[['data_inspecao', LAT_COL, LON_COL, 'total_aedes_aegypti']].copy()
    df_limpo = df_limpo.dropna(subset=['data_inspecao', LAT_COL, LON_COL])

    if verbose:
        print(f"   {tecnica}/seed={seed}: {len(df_limpo):,} registros, "
              f"{df_limpo['total_aedes_aegypti'].sum():,} Aedes")

    return df_limpo


# ─── Validação rápida da função ────────────────────────────────
print("🧪 Teste rápido: carregar 'original' com seed=0 e 'permutacao' com seed=0\n")

df_orig_s0 = carregar_mosquito_tecnica_seed("original", seed=0, verbose=True)
df_perm_s0 = carregar_mosquito_tecnica_seed("permutacao", seed=0, verbose=True)
df_perm_s5 = carregar_mosquito_tecnica_seed("permutacao", seed=5, verbose=True)

# Sanity: original e permutacao devem ter o mesmo número de registros
assert len(df_orig_s0) == len(df_perm_s0), "Original e permutação têm tamanhos diferentes!"

# Sanity: seeds diferentes da permutacao devem gerar coordenadas diferentes
diff_lat = (df_perm_s0[LAT_COL].values != df_perm_s5[LAT_COL].values).sum()
print(f"\n   Diferença entre seed=0 e seed=5 da permutação: {diff_lat:,} pontos com lat diferente")
assert diff_lat > 0, "Seeds diferentes da permutação produzem coordenadas idênticas — bug!"

print(f"\n✅ Função de carregamento OK.")

🧪 Teste rápido: carregar 'original' com seed=0 e 'permutacao' com seed=0

   original/seed=0: 224,194 registros, 69,970 Aedes
   permutacao/seed=0: 224,194 registros, 69,970 Aedes
   permutacao/seed=5: 224,194 registros, 69,970 Aedes

   Diferença entre seed=0 e seed=5 da permutação: 222,182 pontos com lat diferente

✅ Função de carregamento OK.


In [18]:
# ============================================================
# Funções: K-Means de regiões + agregação semanal
# (chamadas pelo loop principal, para cada técnica × seed)
# ============================================================

def montar_regioes_kmeans(df_mosquito, n_regioes=N_REGIOES, random_state=42):
    """
    Agrupa armadilhas em N regiões espaciais via K-Means.

    Parâmetros:
        df_mosquito : DataFrame com colunas latitude, longitude
        n_regioes   : número de clusters (default 15)
        random_state: seed do K-Means (FIXO em 42 para reprodutibilidade
                     da clusterização — não confundir com seed da técnica)

    Retorna:
        df_com_regiao : DataFrame original + coluna 'regiao_cluster'
    """
    # Coordenadas únicas (uma por armadilha/local)
    df_geo_unique = df_mosquito[[LAT_COL, LON_COL]].drop_duplicates().copy()

    n_pontos = len(df_geo_unique)
    if n_pontos < n_regioes:
        raise RuntimeError(
            f"Apenas {n_pontos} pontos únicos disponíveis (< {n_regioes} clusters). "
            f"Provavelmente uma generalização com casas decimais muito agressiva."
        )

    # Normaliza coordenadas antes do K-Means
    scaler = StandardScaler()
    coords_scaled = scaler.fit_transform(df_geo_unique[[LAT_COL, LON_COL]])

    # K-Means com n_init=10 para estabilidade
    km = KMeans(n_clusters=n_regioes, random_state=random_state, n_init=10)
    df_geo_unique['regiao_cluster'] = km.fit_predict(coords_scaled)

    # Merge: adiciona região em todas as inspeções
    df_resultado = pd.merge(
        df_mosquito,
        df_geo_unique[[LAT_COL, LON_COL, 'regiao_cluster']],
        on=[LAT_COL, LON_COL], how='left'
    )

    return df_resultado


def montar_master_weekly(df_com_regiao, df_clima_features):
    """
    Constrói o DataFrame semanal com:
        - X_total_cidade        : total de Aedes na semana, cidade inteira
        - X_regiao_<id>_qtd     : total de Aedes na semana, por região
        - clima_*               : features climáticas (já semanais)

    Parâmetros:
        df_com_regiao    : DataFrame com data_inspecao, lat, lon, total_aedes_aegypti, regiao_cluster
        df_clima_features: DataFrame semanal de features climáticas

    Retorna:
        df_master_weekly : DataFrame indexado por semana, pronto para criar lags
    """
    df = df_com_regiao.copy()
    df = df.set_index('data_inspecao').sort_index()

    # Total por semana (cidade)
    weekly_total = df['total_aedes_aegypti'].resample('W').sum()

    # Total por semana × região
    weekly_regiao = (
        df.groupby([pd.Grouper(freq='W'), 'regiao_cluster'])['total_aedes_aegypti']
        .sum()
        .unstack(fill_value=0)
    )
    weekly_regiao.columns = [f"X_regiao_{int(c)}_qtd" for c in weekly_regiao.columns]

    # Junta tudo
    df_master = pd.DataFrame(index=weekly_total.index)
    df_master['X_total_cidade'] = weekly_total
    df_master = df_master.join(weekly_regiao)

    # Garante presença de todas as 15 regiões mesmo se alguma não teve dados
    for r in range(N_REGIOES):
        col = f"X_regiao_{r}_qtd"
        if col not in df_master.columns:
            df_master[col] = 0

    # Reordena colunas
    cols_regiao = sorted([c for c in df_master.columns if c.startswith('X_regiao_')])
    df_master = df_master[['X_total_cidade'] + cols_regiao]

    # Junta features climáticas (por semana, mesma indexação)
    df_master = df_master.join(df_clima_features, how='left')

    return df_master


# ─── Validação rápida ──────────────────────────────────────────
print("🧪 Teste: pipeline completo até master_weekly para 'original' seed=0\n")

df_orig = carregar_mosquito_tecnica_seed("original", seed=0, verbose=True)
df_orig_reg = montar_regioes_kmeans(df_orig)
df_master = montar_master_weekly(df_orig_reg, df_climate_features_weekly)

print(f"\n📊 DataFrame master_weekly:")
print(f"   Shape: {df_master.shape}")
print(f"   Período: {df_master.index.min().date()} → {df_master.index.max().date()}")
print(f"   Colunas: {list(df_master.columns)[:8]}...")
print(f"\n   Primeiras 3 linhas:")
print(df_master.head(3))

print(f"\n   Total Aedes (verificação cruzada): {df_master['X_total_cidade'].sum():,.0f}")
print(f"   (Deve bater com 69,970 da contagem original)")

# Verifica integridade das features climáticas
clima_cols = [c for c in df_master.columns if c.startswith('clima_')]
print(f"\n   Features climáticas presentes: {clima_cols}")
nan_clima = df_master[clima_cols].isna().sum().sum()
print(f"   Total de NaN em features climáticas: {nan_clima}")

🧪 Teste: pipeline completo até master_weekly para 'original' seed=0

   original/seed=0: 224,194 registros, 69,970 Aedes

📊 DataFrame master_weekly:
   Shape: (232, 20)
   Período: 2019-01-06 → 2023-06-11
   Colunas: ['X_total_cidade', 'X_regiao_0_qtd', 'X_regiao_10_qtd', 'X_regiao_11_qtd', 'X_regiao_12_qtd', 'X_regiao_13_qtd', 'X_regiao_14_qtd', 'X_regiao_1_qtd']...

   Primeiras 3 linhas:
               X_total_cidade  X_regiao_0_qtd  X_regiao_10_qtd  \
data_inspecao                                                    
2019-01-06                 55             0.0              0.0   
2019-01-13                 57             0.0              0.0   
2019-01-20                455            21.0             42.0   

               X_regiao_11_qtd  X_regiao_12_qtd  X_regiao_13_qtd  \
data_inspecao                                                      
2019-01-06                 0.0              5.0              0.0   
2019-01-13                 0.0              1.0              0.0   
201

In [19]:
# ============================================================
# Função: criar lags + médias móveis a partir do master_weekly
# (target Y_surto é definido no momento do treino, dentro do fold,
#  para evitar leakage temporal via percentil global)
# ============================================================

def criar_features_lag(df_master, lags=(2, 3, 4), incluir_mm_4sem=False):
    """
    Cria features defasadas (lags) a partir das colunas X_* e clima_*.

    Política para evitar vazamento temporal:
        - Lag 1 NÃO é criado (semana imediatamente anterior poderia
          vazar informação do target da semana corrente).
        - Lags >= 2 são seguros: representam observações de pelo menos
          2 semanas atrás.

    Parâmetros:
        df_master       : DataFrame de master_weekly
        lags            : tupla de lags a criar (default 2, 3, 4)
        incluir_mm_4sem : se True, cria também médias móveis 4 semanas
                          (com shift de 1 para evitar leakage). Default
                          False — desliguei porque as médias móveis
                          podem mascarar o efeito dos lags individuais.

    Retorna:
        df_features : DataFrame com lags adicionados e NaN removidos.
                      Inclui features_lag (lista de nomes) como atributo.
    """
    df = df_master.copy()

    # Features-base que receberão lags
    feature_cols = [c for c in df.columns if c.startswith('X_') or c.startswith('clima_')]

    # Cria lags
    for col in feature_cols:
        for lag in lags:
            df[f'{col}_lag_{lag}'] = df[col].shift(lag)

    # Médias móveis (opcional)
    if incluir_mm_4sem:
        for col in feature_cols:
            df[f'{col}_mm_4sem'] = df[col].shift(1).rolling(window=4).mean()

    # Remove linhas com NaN (primeiras max(lags) semanas)
    df = df.dropna()

    # Lista final de features para o modelo (apenas lags, sem as variáveis-base)
    features_lag = [c for c in df.columns
                    if (c.startswith('X_') or c.startswith('clima_'))
                    and '_lag_' in c]

    return df, features_lag


def calcular_limiar_surto(df_treino, percentil=80, col='X_total_cidade'):
    """
    Calcula o limiar de surto como percentil 80 da coluna X_total_cidade
    APENAS sobre os dados de treino do fold corrente.

    Crítico: nunca use o dataset inteiro para calcular o limiar — isso
    introduziria vazamento temporal entre treino e teste.
    """
    return df_treino[col].quantile(percentil / 100.0)


def aplicar_target(df, limiar, col='X_total_cidade'):
    """
    Cria a coluna Y_surto_na_semana usando um limiar pré-calculado.
    """
    df = df.copy()
    df['Y_surto_na_semana'] = (df[col] > limiar).astype(int)
    return df


# ─── Validação rápida ──────────────────────────────────────────
print("🧪 Teste: pipeline completo com lags para 'original' seed=0\n")

df_orig = carregar_mosquito_tecnica_seed("original", seed=0, verbose=False)
df_orig_reg = montar_regioes_kmeans(df_orig)
df_master = montar_master_weekly(df_orig_reg, df_climate_features_weekly)

# Aplica lags
df_features, features_lag = criar_features_lag(df_master)

print(f"📊 Após lags:")
print(f"   Shape: {df_features.shape}")
print(f"   Período: {df_features.index.min().date()} → {df_features.index.max().date()}")
print(f"   Features de lag criadas: {len(features_lag)}")
print(f"   (Esperado: 20 colunas-base × 3 lags = 60 features)")

# Mostra algumas features
print(f"\n   Exemplos de features:")
print(f"      Mosquito: {[f for f in features_lag if f.startswith('X_')][:5]}")
print(f"      Clima:    {[f for f in features_lag if f.startswith('clima_')][:5]}")

# Simula um fold do TimeSeriesSplit para testar o cálculo de limiar
n = len(df_features)
n_treino = int(n * 0.7)
df_treino = df_features.iloc[:n_treino]
df_teste = df_features.iloc[n_treino:]

limiar = calcular_limiar_surto(df_treino)
df_treino_com_target = aplicar_target(df_treino, limiar)
df_teste_com_target = aplicar_target(df_teste, limiar)

print(f"\n📊 Simulação fold (70% treino / 30% teste):")
print(f"   Treino: {len(df_treino):,} semanas")
print(f"   Teste:  {len(df_teste):,} semanas")
print(f"   Limiar surto (P80 do treino): {limiar:.1f} Aedes/semana")
print(f"\n   Distribuição target NO TREINO:")
print(df_treino_com_target['Y_surto_na_semana'].value_counts(normalize=True).to_string())
print(f"\n   Distribuição target NO TESTE:")
print(df_teste_com_target['Y_surto_na_semana'].value_counts(normalize=True).to_string())

# Sanity check: ratio de positivos no treino deve ser ~20% (P80)
ratio_treino = df_treino_com_target['Y_surto_na_semana'].mean()
print(f"\n✅ Ratio positivos treino: {ratio_treino:.1%} (esperado ~20%)")
assert 0.15 < ratio_treino < 0.25, f"Ratio fora do esperado: {ratio_treino:.1%}"

🧪 Teste: pipeline completo com lags para 'original' seed=0

📊 Após lags:
   Shape: (224, 80)
   Período: 2019-02-03 → 2023-06-11
   Features de lag criadas: 60
   (Esperado: 20 colunas-base × 3 lags = 60 features)

   Exemplos de features:
      Mosquito: ['X_total_cidade_lag_2', 'X_total_cidade_lag_3', 'X_total_cidade_lag_4', 'X_regiao_0_qtd_lag_2', 'X_regiao_0_qtd_lag_3']
      Clima:    ['clima_chuva_acum_21d_lag_2', 'clima_chuva_acum_21d_lag_3', 'clima_chuva_acum_21d_lag_4', 'clima_temp_min_media_14d_lag_2', 'clima_temp_min_media_14d_lag_3']

📊 Simulação fold (70% treino / 30% teste):
   Treino: 156 semanas
   Teste:  68 semanas
   Limiar surto (P80 do treino): 443.0 Aedes/semana

   Distribuição target NO TREINO:
Y_surto_na_semana
0    0.801282
1    0.198718

   Distribuição target NO TESTE:
Y_surto_na_semana
1    0.514706
0    0.485294

✅ Ratio positivos treino: 19.9% (esperado ~20%)


In [20]:
# ============================================================
# TimeSeriesSplit + funções de treino para 3 classificadores
# ============================================================

def gerar_folds_timeseries(df_features, n_splits=N_FOLDS, min_train_size=80):
    """
    Gera índices de split temporal expansivo.

    Cada fold:
        - Treino: do início até o ponto t_k
        - Teste:  de t_k até t_{k+1}

    Onde t_0 corresponde ao primeiro tamanho de treino válido
    (min_train_size semanas) e t_{N_FOLDS} é o fim da série.

    Parâmetros:
        df_features    : DataFrame com lags já criados, indexado por semana
        n_splits       : número de folds (default 5)
        min_train_size : mínimo de semanas no primeiro treino (default 80,
                         ~ 1.5 anos, suficiente para o modelo aprender padrões
                         sazonais)

    Yield:
        (idx_treino, idx_teste, fold_id) — índices DENTRO de df_features
    """
    n = len(df_features)

    # Tamanho médio de cada janela de teste
    janela_teste = (n - min_train_size) // n_splits

    if janela_teste < 10:
        raise ValueError(
            f"Janela de teste muito pequena ({janela_teste} semanas). "
            f"Reduza n_splits ou min_train_size."
        )

    for fold_id in range(n_splits):
        fim_treino = min_train_size + fold_id * janela_teste
        fim_teste  = min_train_size + (fold_id + 1) * janela_teste

        # No último fold, estende teste até o fim da série
        if fold_id == n_splits - 1:
            fim_teste = n

        idx_treino = np.arange(0, fim_treino)
        idx_teste  = np.arange(fim_treino, fim_teste)

        yield idx_treino, idx_teste, fold_id


# ─── Funções de treino para cada classificador ─────────────────

def treinar_xgboost(X_train, y_train, random_state=42):
    """Treina XGBoost com hiperparâmetros do paper original."""
    scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    model = xgb.XGBClassifier(
        n_estimators=150,
        learning_rate=0.05,
        random_state=random_state,
        n_jobs=-1,
        scale_pos_weight=scale_pos,
        verbosity=0,
    )
    model.fit(X_train, y_train)
    return model


def treinar_random_forest(X_train, y_train, random_state=42):
    """Treina Random Forest balanceado por classe."""
    model = RandomForestClassifier(
        n_estimators=150,
        max_depth=10,
        class_weight='balanced',
        random_state=random_state,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    return model


def treinar_logistic_regression(X_train, y_train, random_state=42):
    """Treina Regressão Logística com regularização L2 e classe balanceada."""
    model = LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        random_state=random_state,
        solver='lbfgs',
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    return model


# Dicionário de classificadores (chave = nome usado no loop)
CLF_FUNCS = {
    'xgboost':             treinar_xgboost,
    'random_forest':       treinar_random_forest,
    'logistic_regression': treinar_logistic_regression,
}


def calcular_metricas(y_true, y_pred, y_score):
    """
    Calcula todas as métricas reportadas no paper.

    Parâmetros:
        y_true  : array-like de labels verdadeiros (0/1)
        y_pred  : array-like de labels preditos    (0/1)
        y_score : array-like de scores [0,1] do classificador
                  (usado para AUC)

    Retorna:
        dict com chaves: auc, recall, precision, f1, balanced_accuracy
    """
    return {
        'auc':                roc_auc_score(y_true, y_score) if len(set(y_true)) > 1 else np.nan,
        'recall':             recall_score(y_true, y_pred, zero_division=0),
        'precision':          precision_score(y_true, y_pred, zero_division=0),
        'f1':                 f1_score(y_true, y_pred, zero_division=0),
        'balanced_accuracy':  balanced_accuracy_score(y_true, y_pred),
    }


# ─── Validação rápida: rodar 1 fold para 'original' com XGBoost ────
print("🧪 Teste: 1 fold com XGBoost para 'original' seed=0\n")

# Pipeline completo
df_orig = carregar_mosquito_tecnica_seed("original", seed=0, verbose=False)
df_orig_reg = montar_regioes_kmeans(df_orig)
df_master = montar_master_weekly(df_orig_reg, df_climate_features_weekly)
df_features, features_lag = criar_features_lag(df_master)

# Gera o primeiro fold
folds = list(gerar_folds_timeseries(df_features))
print(f"📊 {len(folds)} folds gerados:")
for idx_tr, idx_te, k in folds:
    print(f"   Fold {k+1}: treino [{idx_tr.min():3d}:{idx_tr.max():3d}] "
          f"({len(idx_tr):3d} semanas) | teste [{idx_te.min():3d}:{idx_te.max():3d}] "
          f"({len(idx_te):3d} semanas)")

# Roda o primeiro fold com XGBoost
idx_tr, idx_te, _ = folds[0]
df_tr = df_features.iloc[idx_tr]
df_te = df_features.iloc[idx_te]

# Calcula limiar usando só treino
limiar = calcular_limiar_surto(df_tr)
df_tr = aplicar_target(df_tr, limiar)
df_te = aplicar_target(df_te, limiar)

# Separa X e y
X_tr = df_tr[features_lag].values
y_tr = df_tr['Y_surto_na_semana'].values
X_te = df_te[features_lag].values
y_te = df_te['Y_surto_na_semana'].values

# Normaliza
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

# Treina e prediz
modelo = treinar_xgboost(X_tr_s, y_tr)
y_pred = modelo.predict(X_te_s)
y_score = modelo.predict_proba(X_te_s)[:, 1]

metricas = calcular_metricas(y_te, y_pred, y_score)

print(f"\n📊 Resultado fold 1, XGBoost, original:")
print(f"   Treino: {len(y_tr)} sem ({y_tr.mean():.1%} positivos, limiar={limiar:.1f})")
print(f"   Teste:  {len(y_te)} sem ({y_te.mean():.1%} positivos)")
print(f"\n   Métricas:")
for nome, val in metricas.items():
    print(f"      {nome:<22s}: {val:.3f}")

🧪 Teste: 1 fold com XGBoost para 'original' seed=0

📊 5 folds gerados:
   Fold 1: treino [  0: 79] ( 80 semanas) | teste [ 80:107] ( 28 semanas)
   Fold 2: treino [  0:107] (108 semanas) | teste [108:135] ( 28 semanas)
   Fold 3: treino [  0:135] (136 semanas) | teste [136:163] ( 28 semanas)
   Fold 4: treino [  0:163] (164 semanas) | teste [164:191] ( 28 semanas)
   Fold 5: treino [  0:191] (192 semanas) | teste [192:223] ( 32 semanas)

📊 Resultado fold 1, XGBoost, original:
   Treino: 80 sem (20.0% positivos, limiar=447.2)
   Teste:  28 sem (7.1% positivos)

   Métricas:
      auc                   : 0.962
      recall                : 0.000
      precision             : 0.000
      f1                    : 0.000
      balanced_accuracy     : 0.500


In [21]:
# ============================================================
# Loop principal: técnica × seed × classificador × fold
# Salva resultados em /Resultados_Fase2/resultados.parquet
# Reentrante: ao re-rodar, pula combinações já executadas
# ============================================================

from sklearn.metrics import f1_score as f1_fn

CAMINHO_RESULTADOS = f"{BASE_OUT_FASE2}/resultados.parquet"


def otimizar_threshold_f1(y_true, y_scores, grid=None):
    """
    Encontra o threshold que maximiza F1 sobre (y_true, y_scores).

    Usado APENAS no treino para escolher o threshold, depois aplicado
    no teste. Evita threshold default 0.5 que ignora desbalanceamento.

    Parâmetros:
        y_true   : labels verdadeiros (0/1)
        y_scores : probabilidades [0,1] de classe positiva
        grid     : valores de threshold a testar (default: 0.05 a 0.95 em 19 passos)

    Retorna:
        (best_thr, best_f1)
    """
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)

    best_thr, best_f1 = 0.5, -1
    for thr in grid:
        y_pred_thr = (y_scores >= thr).astype(int)
        f1 = f1_fn(y_true, y_pred_thr, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    return best_thr, best_f1


def rodar_um_fold(df_features, features_lag, idx_tr, idx_te, fold_id,
                  clf_nome, clf_func, random_state=42):
    """
    Executa um único fold do TimeSeriesSplit:
        1) Calcula limiar de surto P80 no treino
        2) Treina classificador com threshold default
        3) Otimiza threshold por F1 sobre o próprio treino
        4) Aplica threshold otimizado no teste
        5) Retorna dict com métricas

    Retorna:
        dict com chaves: fold_id, clf, n_treino, n_teste, ratio_pos_treino,
                         ratio_pos_teste, limiar_surto, threshold_otimo,
                         auc, recall, precision, f1, balanced_accuracy
    """
    df_tr = df_features.iloc[idx_tr]
    df_te = df_features.iloc[idx_te]

    # Limiar de surto SOBRE O TREINO (evita leakage temporal)
    limiar = calcular_limiar_surto(df_tr)
    df_tr = aplicar_target(df_tr, limiar)
    df_te = aplicar_target(df_te, limiar)

    # Separa X e y
    X_tr = df_tr[features_lag].values
    y_tr = df_tr['Y_surto_na_semana'].values
    X_te = df_te[features_lag].values
    y_te = df_te['Y_surto_na_semana'].values

    # Normaliza
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    # Treina e prediz (probabilidades)
    modelo = clf_func(X_tr_s, y_tr, random_state=random_state)
    y_scores_tr = modelo.predict_proba(X_tr_s)[:, 1]
    y_scores_te = modelo.predict_proba(X_te_s)[:, 1]

    # Otimiza threshold no TREINO
    thr_otimo, _ = otimizar_threshold_f1(y_tr, y_scores_tr)

    # Aplica threshold otimizado no teste
    y_pred_te = (y_scores_te >= thr_otimo).astype(int)

    # Métricas no teste
    metricas = calcular_metricas(y_te, y_pred_te, y_scores_te)

    return {
        'fold_id':           fold_id,
        'clf':               clf_nome,
        'n_treino':          len(y_tr),
        'n_teste':           len(y_te),
        'ratio_pos_treino':  float(y_tr.mean()),
        'ratio_pos_teste':   float(y_te.mean()),
        'limiar_surto':      float(limiar),
        'threshold_otimo':   float(thr_otimo),
        **metricas,
    }


# ────────────────────────────────────────────────────────────────
# Loop principal
# ────────────────────────────────────────────────────────────────

# Recupera resultados anteriores se existirem (reentrante)
if os.path.exists(CAMINHO_RESULTADOS):
    df_resultados_prev = pd.read_parquet(CAMINHO_RESULTADOS)
    combs_feitas = set(
        zip(df_resultados_prev['tecnica'], df_resultados_prev['seed'], df_resultados_prev['clf'])
    )
    print(f"📂 Resumindo de checkpoint: {len(combs_feitas)} combinações (técnica×seed×clf) já feitas")
else:
    df_resultados_prev = pd.DataFrame()
    combs_feitas = set()
    print(f"🆕 Nenhum checkpoint encontrado. Começando do zero.")

# Monta lista de combinações a executar
combinacoes = []
for tecnica in TECNICAS:
    n_seeds_tec = N_SEEDS_USAR if tecnica in TECNICAS_ESTOCASTICAS else 1
    for seed in range(n_seeds_tec):
        for clf_nome in CLASSIFICADORES:
            if (tecnica, seed, clf_nome) not in combs_feitas:
                combinacoes.append((tecnica, seed, clf_nome))

print(f"\n🔢 Combinações pendentes: {len(combinacoes)}")
print(f"   (técnica × seed × classificador, cada uma roda 5 folds)")
print(f"   Total estimado: ~{len(combinacoes) * 5}  fits")
print(f"\n⏱️  Tempo estimado: ~{len(combinacoes) * 5 * 4 / 60:.0f} min")

# Pré-cacheia features para cada (técnica, seed) — economia importante!
print(f"\n🚀 Iniciando loop principal...\n")

resultados_novos = []

# Para evitar reprocessar features para o mesmo (técnica, seed) quando
# trocamos só o classificador, agrupamos por (técnica, seed) primeiro
combs_por_dados = {}
for tecnica, seed, clf_nome in combinacoes:
    combs_por_dados.setdefault((tecnica, seed), []).append(clf_nome)

# Barra de progresso
pbar = tqdm(combs_por_dados.items(), desc="Pipeline", total=len(combs_por_dados))

for (tecnica, seed), clfs_para_rodar in pbar:
    pbar.set_postfix({'tecnica': tecnica[:20], 'seed': seed})

    # ─── Pré-processamento (1x por técnica×seed) ────────────────
    try:
        df_mosq = carregar_mosquito_tecnica_seed(tecnica, seed)
        df_reg = montar_regioes_kmeans(df_mosq)
        df_master = montar_master_weekly(df_reg, df_climate_features_weekly)
        df_features, features_lag = criar_features_lag(df_master)
        folds = list(gerar_folds_timeseries(df_features))
    except Exception as e:
        print(f"\n⚠️  Erro no pré-processamento de {tecnica}/seed={seed}: {e}")
        continue

    # ─── Itera classificadores ────────────────────────────────────
    for clf_nome in clfs_para_rodar:
        clf_func = CLF_FUNCS[clf_nome]
        for idx_tr, idx_te, fold_id in folds:
            try:
                res = rodar_um_fold(
                    df_features, features_lag, idx_tr, idx_te, fold_id,
                    clf_nome, clf_func
                )
                res['tecnica'] = tecnica
                res['seed'] = seed
                resultados_novos.append(res)
            except Exception as e:
                print(f"\n⚠️  Erro fold {fold_id+1} de {tecnica}/seed={seed}/{clf_nome}: {e}")

    # Checkpoint a cada técnica×seed completa
    if resultados_novos:
        df_novos = pd.DataFrame(resultados_novos)
        df_full = pd.concat([df_resultados_prev, df_novos], ignore_index=True)
        df_full.to_parquet(CAMINHO_RESULTADOS, index=False)

pbar.close()

# Recarrega o resultado final
df_resultados = pd.read_parquet(CAMINHO_RESULTADOS)

print(f"\n✅ Loop concluído!")
print(f"   Total de fits realizados: {len(df_resultados)}")
print(f"   Combinações únicas (técnica × seed × clf): {df_resultados[['tecnica','seed','clf']].drop_duplicates().shape[0]}")
print(f"   Resultados salvos em: {CAMINHO_RESULTADOS}")

# Sumário rápido
print(f"\n📊 Resumo por técnica (mediana AUC sobre seeds × folds × clf):")
sumario = df_resultados.groupby('tecnica')['auc'].median().sort_values(ascending=False)
print(sumario.to_string())

📂 Resumindo de checkpoint: 75 combinações (técnica×seed×clf) já feitas

🔢 Combinações pendentes: 201
   (técnica × seed × classificador, cada uma roda 5 folds)
   Total estimado: ~1005  fits

⏱️  Tempo estimado: ~67 min

🚀 Iniciando loop principal...



Pipeline:   0%|          | 0/67 [00:00<?, ?it/s]


✅ Loop concluído!
   Total de fits realizados: 1380
   Combinações únicas (técnica × seed × clf): 276
   Resultados salvos em: /content/drive/MyDrive/Mestrado/Resultados_Fase2/resultados.parquet

📊 Resumo por técnica (mediana AUC sobre seeds × folds × clf):
tecnica
original              0.918367
microagregacao_k10    0.903846
microagregacao_k5     0.884615
permutacao            0.884615
dp_eps_5.0            0.879432
dp_eps_0.5            0.874510
microagregacao_k2     0.872874
dp_eps_2.0            0.870668
dp_eps_1.0            0.870588
generalizacao_dec2    0.870588
dp_eps_0.1            0.868627


In [25]:
# ============================================================
# Análise estatística: IC bootstrap, Wilcoxon pareado, Holm-Bonferroni
# (versão corrigida do bootstrap_ic)
# ============================================================

from scipy import stats

# Recarrega resultados se necessário
df_res = pd.read_parquet(CAMINHO_RESULTADOS)
print(f"📊 Carregando {len(df_res)} fits para análise estatística\n")

# ────────────────────────────────────────────────────────────────
# 1. IC bootstrap 95% para cada (técnica, métrica)
# ────────────────────────────────────────────────────────────────

def bootstrap_ic(valores, n_boot=N_BOOTSTRAP, conf=0.95):
    """
    Calcula IC via bootstrap (sampling com reposição).

    Retorna (mediana, lower, upper) baseado nos percentis (1-conf)/2 e
    1-(1-conf)/2. NaN values são removidos antes do bootstrap.
    """
    valores = np.array(valores)
    valores = valores[~np.isnan(valores)]  # remove NaN
    if len(valores) < 3:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(42)
    boot_medianas = np.empty(n_boot)
    for i in range(n_boot):
        sample = rng.choice(valores, size=len(valores), replace=True)
        boot_medianas[i] = np.median(sample)

    # Percentis para IC: (1-conf)/2 e 1-(1-conf)/2
    alpha = 1 - conf
    lower_pct = (alpha / 2) * 100        # ex: 2.5
    upper_pct = (1 - alpha / 2) * 100    # ex: 97.5

    lower = np.percentile(boot_medianas, lower_pct)
    upper = np.percentile(boot_medianas, upper_pct)

    return float(np.median(valores)), float(lower), float(upper)


METRICAS = ['auc', 'recall', 'precision', 'f1', 'balanced_accuracy']

print("🔬 Calculando IC bootstrap 95% por (técnica, métrica)...")

resultados_ic = []
for tecnica in df_res['tecnica'].unique():
    sub = df_res[df_res['tecnica'] == tecnica]
    for metrica in METRICAS:
        mediana, lower, upper = bootstrap_ic(sub[metrica].values)
        resultados_ic.append({
            'tecnica': tecnica,
            'metrica': metrica,
            'mediana': mediana,
            'ic_lower': lower,
            'ic_upper': upper,
            'n_amostras': sub[metrica].notna().sum(),
        })

df_ic = pd.DataFrame(resultados_ic)
print(f"   ✅ {len(df_ic)} IC calculados\n")

# Pivotear para tabela limpa: técnica × métrica com mediana [IC]
print("📊 Mediana [IC 95%] por técnica × métrica:\n")
for metrica in METRICAS:
    print(f"\n--- {metrica.upper()} ---")
    sub = df_ic[df_ic['metrica'] == metrica].sort_values('mediana', ascending=False)
    for _, row in sub.iterrows():
        print(f"  {row['tecnica']:<25s} {row['mediana']:.3f} "
              f"[{row['ic_lower']:.3f}; {row['ic_upper']:.3f}]")

# ────────────────────────────────────────────────────────────────
# 2. Wilcoxon pareado vs baseline (original)
# ────────────────────────────────────────────────────────────────

print("\n\n🔬 Wilcoxon pareado: cada técnica vs 'original' (por métrica)\n")

resultados_wilcoxon = []

for metrica in METRICAS:
    print(f"\n--- {metrica.upper()} (Wilcoxon vs original) ---")

    # Para Wilcoxon pareado: agrupa por (fold_id, clf), tira mediana entre seeds
    sub_orig = df_res[df_res['tecnica'] == 'original']
    med_orig = (sub_orig.groupby(['fold_id', 'clf'])[metrica]
                .median().reset_index())

    p_valores = {}
    for tecnica in df_res['tecnica'].unique():
        if tecnica == 'original':
            continue
        sub_tec = df_res[df_res['tecnica'] == tecnica]
        med_tec = (sub_tec.groupby(['fold_id', 'clf'])[metrica]
                   .median().reset_index())

        # Merge para garantir pareamento perfeito
        m = med_tec.merge(med_orig, on=['fold_id', 'clf'], suffixes=('_tec', '_orig'))
        m = m.dropna(subset=[f'{metrica}_tec', f'{metrica}_orig'])

        if len(m) < 5:
            p_valores[tecnica] = np.nan
            continue

        # Wilcoxon pareado
        try:
            diff = m[f'{metrica}_tec'].values - m[f'{metrica}_orig'].values
            # Se todas as diferenças são zero, Wilcoxon falha
            if np.all(diff == 0):
                p_valores[tecnica] = 1.0
            else:
                _, p = stats.wilcoxon(m[f'{metrica}_tec'], m[f'{metrica}_orig'],
                                       zero_method='wilcox', alternative='two-sided')
                p_valores[tecnica] = p
        except Exception as e:
            p_valores[tecnica] = np.nan

    # Holm-Bonferroni
    tecnicas_ordenadas = sorted(
        p_valores.keys(),
        key=lambda t: p_valores[t] if not np.isnan(p_valores[t]) else 1.0
    )
    n_comparacoes = sum(1 for p in p_valores.values() if not np.isnan(p))

    p_ajustados = {}
    for rank, tecnica in enumerate(tecnicas_ordenadas):
        p_bruto = p_valores[tecnica]
        if np.isnan(p_bruto):
            p_ajustados[tecnica] = np.nan
            continue
        p_ajustados[tecnica] = min(1.0, (n_comparacoes - rank) * p_bruto)

    # Imprime
    print(f"  {'Técnica':<25s} {'p bruto':>10s} {'p Holm':>10s} {'Sig.':>5s}")
    for tecnica in tecnicas_ordenadas:
        p_b = p_valores[tecnica]
        p_a = p_ajustados[tecnica]
        sig = '***' if (not np.isnan(p_a) and p_a < 0.001) else \
              '**'  if (not np.isnan(p_a) and p_a < 0.01)  else \
              '*'   if (not np.isnan(p_a) and p_a < 0.05)  else 'n.s.'
        p_b_str = f"{p_b:.4f}" if not np.isnan(p_b) else 'NaN'
        p_a_str = f"{p_a:.4f}" if not np.isnan(p_a) else 'NaN'
        print(f"  {tecnica:<25s} {p_b_str:>10s} {p_a_str:>10s} {sig:>5s}")

        resultados_wilcoxon.append({
            'metrica': metrica,
            'tecnica': tecnica,
            'p_bruto': p_b,
            'p_holm': p_a,
            'significativo_5pct': (not np.isnan(p_a) and p_a < 0.05),
        })

df_wilcoxon = pd.DataFrame(resultados_wilcoxon)

# Salva tudo
df_ic.to_parquet(f"{BASE_OUT_FASE2}/ic_por_tecnica.parquet", index=False)
df_wilcoxon.to_parquet(f"{BASE_OUT_FASE2}/wilcoxon_vs_original.parquet", index=False)

print(f"\n✅ Análise estatística concluída.")
print(f"   IC salvos em: {BASE_OUT_FASE2}/ic_por_tecnica.parquet")
print(f"   Wilcoxon salvos em: {BASE_OUT_FASE2}/wilcoxon_vs_original.parquet")

# Resumo final
print(f"\n📋 RESUMO:")
n_sig_auc = df_wilcoxon[df_wilcoxon['metrica'] == 'auc']['significativo_5pct'].sum()
n_total_auc = len(df_wilcoxon[df_wilcoxon['metrica'] == 'auc'])
print(f"   AUC: {n_sig_auc}/{n_total_auc} técnicas diferem significativamente do original (p<0.05 após Holm)")

📊 Carregando 1380 fits para análise estatística

🔬 Calculando IC bootstrap 95% por (técnica, métrica)...
   ✅ 55 IC calculados

📊 Mediana [IC 95%] por técnica × métrica:


--- AUC ---
  original                  0.918 [0.803; 0.973]
  microagregacao_k10        0.904 [0.881; 0.925]
  permutacao                0.885 [0.856; 0.923]
  microagregacao_k5         0.885 [0.862; 0.916]
  dp_eps_5.0                0.879 [0.857; 0.922]
  dp_eps_0.5                0.875 [0.845; 0.913]
  microagregacao_k2         0.873 [0.850; 0.904]
  dp_eps_2.0                0.871 [0.854; 0.901]
  generalizacao_dec2        0.871 [0.830; 0.942]
  dp_eps_1.0                0.871 [0.855; 0.923]
  dp_eps_0.1                0.869 [0.846; 0.906]

--- RECALL ---
  generalizacao_dec2        0.765 [0.231; 0.941]
  original                  0.714 [0.077; 0.824]
  microagregacao_k5         0.710 [0.588; 0.846]
  dp_eps_0.1                0.710 [0.588; 0.846]
  microagregacao_k10        0.710 [0.571; 0.857]
  dp_eps_1.0    